# Neural Ranking Model

Train a small PyTorch ranking model for Stage 2 of the recommendation system.

This notebook:

- loads the processed user data
- reuses interest-vector retrieval to create candidate pairs
- builds pairwise ranking features
- creates proxy relevance labels because the dataset has no real interaction labels
- trains a small neural ranker
- returns top 10 recommendations for a sample user
- saves the trained model artifact and sample recommendations


## Assessment note: proxy labels, not real user preference

This neural ranker is trained on proxy labels because the dataset does not include real engagement outcomes such as follows, clicks, likes, messages, or accepted recommendations.

The model demonstrates the required two-stage recommendation architecture:

- Stage 1 retrieves 100 candidates.
- Stage 2 reranks those candidates with a PyTorch neural model.

However, the current model should not be treated as a production-quality preference model. It learns a synthetic scoring rule based on interest similarity, location match, and age closeness. In production, the same ranking architecture should be retrained using real behavioral labels.


In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Make paths work whether the notebook is run from the project root or from EDA/.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data" / "processed_data.csv").exists() else CURRENT_DIR.parent

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed_data.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "data"
MODEL_PATH = MODEL_DIR / "neural_ranker.pt"
TOP10_OUTPUT_PATH = OUTPUT_DIR / "sample_top10_recommendations.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load processed data

Use the processed dataset from `02_preprocessing.ipynb`. The raw dataset remains untouched.


In [2]:
users_df = pd.read_csv(PROCESSED_DATA_PATH, dtype={"user_id": "string"})

# The interest lists are stored as JSON strings in the processed CSV.
users_df["interests"] = users_df["interests_deduped"].apply(json.loads)
users_df["age"] = users_df["age"].astype(float)

users_df.head()


,user_id,name,gender,dob,age,city,country,city_country_key,interests_raw,interests_list,interests_deduped,interest_count,split,interests
0,00001,Jesse Lawhorn,Female,1958-10-15,67.0,Sibolga,Indonesia,"Sibolga, Indonesia","'Movies', 'Fashion', 'Fashion', 'Books'","[""Movies"", ""Fashion"", ""Fashion"", ""Books""]","[""Movies"", ""Fashion"", ""Books""]",3,train,"[Movies, Fashion, Books]"
1,00002,Stacy Payne,Female,2004-07-21,22.0,Al Abyār,Libya,"Al Abyār, Libya","'Gaming', 'Finance and investments', 'Outdoor ...","[""Gaming"", ""Finance and investments"", ""Outdoor...","[""Gaming"", ""Finance and investments"", ""Outdoor...",4,train,"[Gaming, Finance and investments, Outdoor acti..."
2,00003,Katrina Nicewander,Female,2000-02-07,26.0,Wādī as Sīr,Jordan,"Wādī as Sīr, Jordan","'DIY and crafts', 'Music', 'Science', 'Fashion'","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]",4,validation,"[DIY and crafts, Music, Science, Fashion]"
3,00004,Eric Yarbrough,Male,1985-04-14,41.0,Matera,Italy,"Matera, Italy","'Outdoor activities', 'Cars and automobiles'","[""Outdoor activities"", ""Cars and automobiles""]","[""Outdoor activities"", ""Cars and automobiles""]",2,train,"[Outdoor activities, Cars and automobiles]"
4,00005,Daniel Adkins,Female,1955-09-18,70.0,Biruaca,Venezuela,"Biruaca, Venezuela","'Politics', 'History'","[""Politics"", ""History""]","[""Politics"", ""History""]",2,train,"[Politics, History]"


## 2. Build interest vectors for candidate retrieval

The neural ranker does not score every possible user pair. We first retrieve a smaller candidate pool using nearest-neighbor search over interest vectors.


In [3]:
interest_vocab = sorted({interest for interests in users_df["interests"] for interest in interests})
interest_to_idx = {interest: idx for idx, interest in enumerate(interest_vocab)}

interest_matrix = np.zeros((len(users_df), len(interest_vocab)), dtype=np.float32)
for row_idx, interests in enumerate(users_df["interests"]):
    for interest in interests:
        interest_matrix[row_idx, interest_to_idx[interest]] = 1.0

ann_index = NearestNeighbors(
    n_neighbors=101,  # 100 candidates + the target user itself
    metric="cosine",
    algorithm="brute",
)
ann_index.fit(interest_matrix)

pd.DataFrame({
    "users": [len(users_df)],
    "interest_features": [len(interest_vocab)],
})


,users,interest_features
0,100000,29


## 3. Candidate retrieval helper

This returns the top candidate users for one target user. The target user is excluded from its own recommendation pool.


In [4]:
def retrieve_candidates(target_user_id: str, top_k: int = 100) -> pd.DataFrame:
    """Retrieve candidate users with similar interest vectors for a target user."""
    target_idx = users_df.index[users_df["user_id"] == target_user_id][0]
    distances, indices = ann_index.kneighbors(interest_matrix[target_idx:target_idx + 1], n_neighbors=top_k + 1)

    candidate_rows = []
    for distance, idx in zip(distances[0], indices[0]):
        candidate_id = users_df.iloc[idx]["user_id"]
        if candidate_id == target_user_id:
            continue

        row = users_df.iloc[idx].copy()
        row["ann_similarity"] = 1 - float(distance)
        candidate_rows.append(row)

        if len(candidate_rows) == top_k:
            break

    return pd.DataFrame(candidate_rows).reset_index(drop=True)


## 4. Pairwise feature builder

These features describe the relationship between a target user and each retrieved candidate.


In [5]:
def build_pairwise_features(target_user: pd.Series, candidates: pd.DataFrame) -> pd.DataFrame:
    """Build ranking features for target-candidate user pairs."""
    target_interests = set(target_user["interests"])
    rows = []

    for _, candidate in candidates.iterrows():
        candidate_interests = set(candidate["interests"])
        overlap_count = len(target_interests & candidate_interests)
        union_count = len(target_interests | candidate_interests)
        jaccard = overlap_count / union_count if union_count else 0.0
        age_difference = abs(float(target_user["age"]) - float(candidate["age"]))
        age_similarity = max(0.0, 1.0 - age_difference / 60.0)

        rows.append({
            "target_user_id": target_user["user_id"],
            "candidate_user_id": candidate["user_id"],
            "candidate_name": candidate["name"],
            "candidate_city": candidate["city"],
            "candidate_country": candidate["country"],
            "candidate_interests": candidate["interests"],
            "ann_similarity": float(candidate["ann_similarity"]),
            "interest_overlap_count": overlap_count,
            "interest_union_count": union_count,
            "interest_jaccard_similarity": jaccard,
            "same_country": int(target_user["country"] == candidate["country"]),
            "same_city_country_key": int(target_user["city_country_key"] == candidate["city_country_key"]),
            "age_difference": age_difference,
            "age_similarity": age_similarity,
            "target_interest_count": len(target_interests),
            "candidate_interest_count": len(candidate_interests),
        })

    return pd.DataFrame(rows)

FEATURE_COLUMNS = [
    "ann_similarity",
    "interest_overlap_count",
    "interest_jaccard_similarity",
    "same_country",
    "same_city_country_key",
    "age_difference",
    "age_similarity",
    "target_interest_count",
    "candidate_interest_count",
]


## 5. Proxy relevance label

The proxy label gives the neural model something to learn. It combines interest similarity, location match, and age closeness.

This is not a real user preference label. It is a training target created only because the dataset does not contain behavioral outcomes.


In [6]:
def add_proxy_label(pair_df: pd.DataFrame) -> pd.DataFrame:
    """Add a synthetic relevance score for training the demonstration ranker."""
    pair_df = pair_df.copy()
    pair_df["proxy_relevance_score"] = (
        0.55 * pair_df["interest_jaccard_similarity"]
        + 0.20 * pair_df["ann_similarity"]
        + 0.10 * pair_df["same_country"]
        + 0.10 * pair_df["same_city_country_key"]
        + 0.05 * pair_df["age_similarity"]
    ).clip(0, 1)
    return pair_df


## 6. Build training and validation pairs

To keep this notebook fast and readable, train on a sample of users instead of generating pairs for the full 100,000-user dataset.


In [7]:
def build_training_pairs(split: str, target_count: int, top_k: int = 100) -> pd.DataFrame:
    """Create target-candidate training rows for a sampled split."""
    split_users = users_df[users_df["split"] == split].sample(
        n=target_count,
        random_state=SEED,
    )

    pair_frames = []
    for _, target_user in split_users.iterrows():
        candidates = retrieve_candidates(target_user["user_id"], top_k=top_k)
        pair_features = build_pairwise_features(target_user, candidates)
        pair_frames.append(pair_features)

    return add_proxy_label(pd.concat(pair_frames, ignore_index=True))

train_pairs = build_training_pairs(split="train", target_count=500, top_k=100)
val_pairs = build_training_pairs(split="validation", target_count=100, top_k=100)

pd.DataFrame({
    "dataset": ["train", "validation"],
    "target_users": [train_pairs["target_user_id"].nunique(), val_pairs["target_user_id"].nunique()],
    "candidate_pairs": [len(train_pairs), len(val_pairs)],
})


,dataset,target_users,candidate_pairs
0,train,500,50000
1,validation,100,10000


## 7. Prepare tensors

Standardize numeric features before feeding them into the neural network.


In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_pairs[FEATURE_COLUMNS]).astype(np.float32)
y_train = train_pairs["proxy_relevance_score"].to_numpy(dtype=np.float32).reshape(-1, 1)

X_val = scaler.transform(val_pairs[FEATURE_COLUMNS]).astype(np.float32)
y_val = val_pairs["proxy_relevance_score"].to_numpy(dtype=np.float32).reshape(-1, 1)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=512,
    shuffle=True,
)

val_tensor = torch.from_numpy(X_val)
val_target = torch.from_numpy(y_val)

pd.DataFrame({
    "feature_count": [X_train.shape[1]],
    "train_rows": [X_train.shape[0]],
    "validation_rows": [X_val.shape[0]],
})


,feature_count,train_rows,validation_rows
0,9,50000,10000


## 8. Train PyTorch neural ranker

The model predicts a relevance score for each target-candidate pair. Higher score means the candidate should rank higher.


In [9]:
class NeuralRanker(nn.Module):
    """Small MLP ranker for target-candidate pair features."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = NeuralRanker(input_dim=len(FEATURE_COLUMNS))
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = []
for epoch in range(1, 21):
    model.train()
    train_losses = []

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        prediction = model(batch_x)
        loss = loss_fn(prediction, batch_y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_prediction = model(val_tensor)
        val_loss = loss_fn(val_prediction, val_target).item()

    history.append({
        "epoch": epoch,
        "train_mse": float(np.mean(train_losses)),
        "validation_mse": val_loss,
    })

history_df = pd.DataFrame(history)
history_df.tail()


,epoch,train_mse,validation_mse
15,16,0.000081,0.000043
16,17,0.000071,0.000044
17,18,0.000065,0.000042
18,19,0.000060,0.000059
19,20,0.000052,0.000060


## 9. Validate model behavior

Since the labels are synthetic, these metrics only show how well the neural ranker learned the proxy relevance function.


In [10]:
model.eval()
with torch.no_grad():
    val_predictions = model(val_tensor).numpy().reshape(-1)

validation_summary = pd.DataFrame({
    "metric": ["validation_mse", "validation_rmse", "validation_mae"],
    "value": [
        mean_squared_error(y_val.reshape(-1), val_predictions),
        np.sqrt(mean_squared_error(y_val.reshape(-1), val_predictions)),
        mean_absolute_error(y_val.reshape(-1), val_predictions),
    ],
})

validation_summary


,metric,value
0,validation_mse,0.000060
1,validation_rmse,0.007714
2,validation_mae,0.005723


## 10. Generate top 10 recommendations for a sample user

Use Stage 1 retrieval to get 100 candidates, then use the neural ranker to rerank them into the final top 10.


In [11]:
target_user = users_df[users_df["split"] == "test"].iloc[0]
target_candidates = retrieve_candidates(target_user["user_id"], top_k=100)
target_pairs = add_proxy_label(build_pairwise_features(target_user, target_candidates))

X_target = scaler.transform(target_pairs[FEATURE_COLUMNS]).astype(np.float32)
with torch.no_grad():
    target_pairs["predicted_relevance_score"] = model(torch.from_numpy(X_target)).numpy().reshape(-1)

top10_recommendations = target_pairs.sort_values(
    by="predicted_relevance_score",
    ascending=False,
).head(10).reset_index(drop=True)

top10_recommendations[
    [
        "candidate_user_id",
        "candidate_name",
        "candidate_city",
        "candidate_country",
        "candidate_interests",
        "predicted_relevance_score",
        "proxy_relevance_score",
        "interest_jaccard_similarity",
        "same_country",
        "same_city_country_key",
        "age_difference",
    ]
]


,candidate_user_id,candidate_name,candidate_city,candidate_country,candidate_interests,predicted_relevance_score,proxy_relevance_score,interest_jaccard_similarity,same_country,same_city_country_key,age_difference
0,14313,Mark Boudreau,Owensboro,United States,"[Photography, History, Books]",0.780227,0.791667,1.000000,0,0,10.0
1,74160,Marisol Turner,Danville,United States,"[Photography, Books, History]",0.771911,0.784167,1.000000,0,0,19.0
2,74128,Marcia Sweene,Chiba,Japan,"[Photography, History, Books]",0.770206,0.782500,1.000000,0,0,21.0
3,28264,Rosa Smith,La Unión,El Salvador,"[Photography, History, Books]",0.768493,0.780833,1.000000,0,0,23.0
4,49817,Ethel Muniz,Bobingen,Germany,"[History, Books, Photography]",0.768493,0.780833,1.000000,0,0,23.0
5,58460,Jared Hines,Vila-real,Spain,"[Photography, History]",0.653537,0.666633,0.666667,1,0,16.0
6,24183,Abraham Freeland,Hāthras,India,"[Technology, Books, History, Photography]",0.630939,0.634038,0.750000,0,0,2.0
7,02911,Peggy Hopkins,Lakshmīpur,Bangladesh,"[Books, History, DIY and crafts, Photography]",0.627616,0.631538,0.750000,0,0,5.0
8,22806,John Davis,Norristown,United States,"[Photography, Gardening, Books, History]",0.620271,0.624872,0.750000,0,0,13.0
9,45018,Felicia Svobodny,Naperville,United States,"[Photography, History, Books, Fashion]",0.616539,0.621538,0.750000,0,0,17.0


## 11. Save artifacts

Save the trained model and sample recommendation output for backend integration.


In [12]:
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_columns": FEATURE_COLUMNS,
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "interest_vocab": interest_vocab,
}, MODEL_PATH)

top10_recommendations.to_csv(TOP10_OUTPUT_PATH, index=False)

pd.DataFrame({
    "artifact": ["model", "sample_top10_recommendations"],
    "path": [str(MODEL_PATH.relative_to(PROJECT_ROOT)), str(TOP10_OUTPUT_PATH.relative_to(PROJECT_ROOT))],
})


,artifact,path
0,model,models/neural_ranker.pt
1,sample_top10_recommendations,data/sample_top10_recommendations.csv


## Neural ranking summary

- Stage 1 retrieves 100 candidates using nearest-neighbor interest retrieval.
- Stage 2 reranks those candidates with a PyTorch neural model.
- The model uses pairwise interest, location, age, and ANN similarity features.
- This notebook uses proxy labels only because the dataset has no real engagement labels.
- The output demonstrates the backend ranking pipeline, not proven production recommendation quality.
- Real production ranking quality would require behavioral labels such as clicks, follows, likes, messages, or accepted recommendations.
